# LocalRetro 数据处理教程

## 概述

本 notebook 展示 LocalRetro（Local Template-Based Retrosynthesis）的**完整数据处理流程**。

LocalRetro 的核心创新在于使用**局部模板**来描述反应变化——不再像 GLN 那样使用一个覆盖整个反应的全局模板，而是将反应变化分解到每个原子/键上，为它们分别标注局部编辑操作。

数据处理流水线：

```
原始反应 SMILES (带原子映射)
  ↓ Step 1: 反应解析与变化原子识别
识别出反应前后发生变化的原子
  ↓ Step 2: 局部模板提取
为每个变化位点提取 atom template 或 bond template
  ↓ Step 3: 模板统计与编号
构建模板词典（atom_templates + bond_templates + template_infos）
  ↓ Step 4: 产物标注
为产物中的每个原子/键分配模板类别标签
  ↓ Step 5: DGL 分子图构建
构建 DGL 图 + WeaveAtomFeaturizer + CanonicalBondFeaturizer
```

### 教学方式

为了便于理解，本 notebook **不依赖 LocalRetro 原始代码的 import**，而是用最小化的纯 Python + RDKit 代码重现每一步的核心逻辑。我们使用一个**微型数据集**（5 条反应）来完整演示整个流程。

---

In [1]:
# ====== 基础导入 ======
import numpy as np
import pandas as pd
import os
from collections import defaultdict, Counter
from pathlib import Path
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, rdChemReactions
from IPython.display import display, HTML

RDLogger.DisableLog('rdApp.*')
print('所有导入成功！')

所有导入成功！


## 构建微型教学数据集

我们读取本教程配置好的 demo 数据。每条反应包含：
- **reactants**: 反应物（带原子映射编号）
- **reagents**: 试剂/催化剂
- **production**: 产物（带原子映射编号）

> **关键区别**: LocalRetro 需要**原子映射编号 (atom map number)**，因为它需要知道反应前后每个原子的对应关系，从而识别哪些原子/键发生了变化。

In [2]:
# ====== 加载微型数据集 ======
demo_data = pd.read_csv('data/demo_data.csv')

print(f'教学数据集包含 {len(demo_data)} 条反应：')
print('=' * 80)
for i, row in demo_data.iterrows():  
    rxn = row['reactants>reagents>production']
    parts = rxn.split('>')
    reactants = parts[0]
    product = parts[-1]
    print(f'\n反应 {i} (类型 {row["class"]}):')
    print(f'  反应物: {reactants}')
    print(f'  产物:   {product}')
    if i >= 4:
        break # 只展示前5条

教学数据集包含 1000 条反应：

反应 0 (类型 nan):
  反应物: C1CCOC1.CCOCC.Cc1ccccc1.O=[CH:10][CH:9]1[C:7](=[O:8])[c:5]2[c:4]([n:3][c:2]([CH3:1])[s:6]2)[CH2:15][CH2:14]1.[NH:11]([CH3:12])[CH3:13]
  产物:   [CH3:1][c:2]1[n:3][c:4]2[c:5]([s:6]1)[C:7](=[O:8])/[C:9](=[CH:10]\[N:11]([CH3:12])[CH3:13])[CH2:14][CH2:15]2

反应 1 (类型 nan):
  反应物: CCN(CC)CC.CCOCC.CN(C)C=O.O=C1CCC(=O)N1O[C:2](=[O:1])[O:11][CH2:12][c:13]1[cH:14][cH:15][cH:16][cH:17][cH:18]1.[NH2:3][C@H:4]1[CH:5]=[CH:6][C@@H:7]([OH:8])[CH2:9][CH2:10]1
  产物:   [O:1]=[C:2]([NH:3][CH:4]1[CH:5]=[CH:6][CH:7]([OH:8])[CH2:9][CH2:10]1)[O:11][CH2:12][c:13]1[cH:14][cH:15][cH:16][cH:17][cH:18]1

反应 2 (类型 nan):
  反应物: Cc1c(CC(=O)Cl)[c:26]2[c:5](n1[C:11](=[O:12])[c:13]1[cH:14][cH:15][c:16](Cl)[cH:17][cH:18]1)[cH:4][cH:3][c:2](O[CH3:1])[cH:27]2.Cc1c(CC(=O)O)c2cc(O[CH3:23])ccc2n1C(=O)[c:19]1[cH:20][cH:21][c:22](Cl)[cH:24][cH:25]1.Cl.ClCCl.O=S(Cl)Cl.[nH:6]1[cH:7][cH:8][n:10][cH:9]1
  产物:   [CH3:1][c:2]1[cH:3][cH:4][c:5]([N:6]=[CH:7]/[CH:8]=[CH:9]/[N:10]([C:11](=[O:12])[c

---

## Step 1: 反应解析与变化原子识别

### 核心思想

LocalRetro 的第一步是比较产物和反应物中对应原子（通过 atom map number 匹配）的性质差异，找出所有**发生变化的原子 (changed atoms)**。

判断原子是否发生变化的依据：
1. **邻居原子不同**: 原子的连接关系改变（断键/成键）
2. **化学键类型不同**: 键的类型改变（单→双 等）
3. **氢原子数量变化**: 显式/隐式 H 数改变
4. **电荷变化**: 形式电荷改变
5. **手性变化**: 立体化学改变

### LocalRetro 源码对应
- 文件: `LocalTemplate/template_extractor.py` → `get_changed_atoms()`
- 辅助: `LocalTemplate/template_extract_utils.py` → `atoms_are_different()`

In [3]:
# ====== Step 1: 识别变化原子 ======
# 对应 LocalRetro 源码: LocalTemplate/template_extractor.py → get_changed_atoms()

def get_tagged_atoms(mol):
    """获取分子中带原子映射编号的原子"""
    atoms = []
    atom_tags = []
    for atom in mol.GetAtoms():
        if atom.HasProp('molAtomMapNumber'):
            atoms.append(atom)
            atom_tags.append(str(atom.GetProp('molAtomMapNumber')))
    return atoms, atom_tags

def atom_neighbors(atom):
    """获取原子的邻居映射编号列表（排序后）"""
    return sorted([n.GetAtomMapNum() for n in atom.GetNeighbors()])

def bond_to_smarts(bond):
    """将键转换为唯一标识字符串"""
    bond_type_map = {'SINGLE': '-', 'DOUBLE': '=', 'TRIPLE': '#', 'AROMATIC': '@'}
    a1 = str(bond.GetBeginAtom().GetAtomicNum())
    a2 = str(bond.GetEndAtom().GetAtomicNum())
    if bond.GetBeginAtom().HasProp('molAtomMapNumber'):
        a1 += bond.GetBeginAtom().GetProp('molAtomMapNumber')
    if bond.GetEndAtom().HasProp('molAtomMapNumber'):
        a2 += bond.GetEndAtom().GetProp('molAtomMapNumber')
    atoms = sorted([a1, a2])
    bt = bond_type_map.get(str(bond.GetBondType()), '?')
    return f'{atoms[0]}{bt}{atoms[1]}'

def atoms_are_different(atom1, atom2):
    """
    比较产物和反应物中对应原子是否发生了变化。
    
    对应 LocalRetro 源码: template_extractor.py → atoms_are_different()
    
    检查项:
    1. 原子类型（理论上不变，但检查防止映射错误）
    2. 自由基电子数
    3. 形式电荷
    4. 氢原子总数
    5. 邻居连接关系（断键/成键的核心判据）
    6. 化学键类型
    """
    if atom1.GetAtomicNum() != atom2.GetAtomicNum():
        return True
    if atom1.GetNumRadicalElectrons() != atom2.GetNumRadicalElectrons():
        return True
    if atom1.GetFormalCharge() != atom2.GetFormalCharge():
        return True
    if atom1.GetTotalNumHs() != atom2.GetTotalNumHs():
        return True
    if atom_neighbors(atom1) != atom_neighbors(atom2):
        return True
    bonds1 = sorted([bond_to_smarts(b) for b in atom1.GetBonds()])
    bonds2 = sorted([bond_to_smarts(b) for b in atom2.GetBonds()])
    if bonds1 != bonds2:
        return True
    return False

def get_changed_atoms(reactants, products):
    """
    找出反应前后发生变化的所有原子。
    
    对应 LocalRetro 源码: template_extractor.py → get_changed_atoms()
    
    返回: (changed_atoms, changed_atom_tags)
    """
    prod_atoms, prod_tags = get_tagged_atoms(products[0] if isinstance(products, list) else products)
    
    # 收集所有反应物原子
    all_reac_atoms = []
    all_reac_tags = []
    for r in (reactants if isinstance(reactants, list) else [reactants]):
        atoms, tags = get_tagged_atoms(r)
        all_reac_atoms.extend(atoms)
        all_reac_tags.extend(tags)
    
    changed_atoms = []
    changed_atom_tags = []
    
    for i, ptag in enumerate(prod_tags):
        for j, rtag in enumerate(all_reac_tags):
            if rtag != ptag:
                continue
            if rtag not in changed_atom_tags:
                if atoms_are_different(prod_atoms[i], all_reac_atoms[j]):
                    changed_atoms.append(all_reac_atoms[j])
                    changed_atom_tags.append(rtag)
                    break
    
    # 反应物中出现但产物中不出现的原子（离去基团）
    for j, rtag in enumerate(all_reac_tags):
        if rtag not in changed_atom_tags and rtag not in prod_tags:
            changed_atoms.append(all_reac_atoms[j])
            changed_atom_tags.append(rtag)
    
    return changed_atoms, changed_atom_tags

In [4]:
# ====== 演示变化原子识别 ======

print('变化原子识别演示')
print('=' * 70)

for i, row in demo_data.iterrows():
    rxn = row['reactants>reagents>production']
    parts = rxn.split('>')
    reactant_smiles = parts[0]
    product_smiles = parts[-1]
    
    # 解析分子
    prod_mol = Chem.MolFromSmiles(product_smiles)
    reac_mols = [Chem.MolFromSmiles(s) for s in reactant_smiles.split('.') if Chem.MolFromSmiles(s) is not None]
    
    # 找变化原子
    changed_atoms, changed_tags = get_changed_atoms(reac_mols, prod_mol)
    
    print(f'\n反应 {i}:')
    print(f'  产物: {product_smiles}')
    print(f'  变化原子映射编号: {changed_tags}')
    for atom, tag in zip(changed_atoms, changed_tags):
        print(f'    原子 {tag}: {atom.GetSymbol()} (度数={atom.GetDegree()}, H数={atom.GetTotalNumHs()})')

变化原子识别演示

反应 0:
  产物: [CH3:1][c:2]1[n:3][c:4]2[c:5]([s:6]1)[C:7](=[O:8])/[C:9](=[CH:10]\[N:11]([CH3:12])[CH3:13])[CH2:14][CH2:15]2
  变化原子映射编号: ['9', '10', '11']
    原子 9: C (度数=3, H数=1)
    原子 10: C (度数=2, H数=1)
    原子 11: N (度数=2, H数=1)

反应 1:
  产物: [O:1]=[C:2]([NH:3][CH:4]1[CH:5]=[CH:6][CH:7]([OH:8])[CH2:9][CH2:10]1)[O:11][CH2:12][c:13]1[cH:14][cH:15][cH:16][cH:17][cH:18]1
  变化原子映射编号: ['2', '3']
    原子 2: C (度数=3, H数=0)
    原子 3: N (度数=1, H数=2)

反应 2:
  产物: [CH3:1][c:2]1[cH:3][cH:4][c:5]([N:6]=[CH:7]/[CH:8]=[CH:9]/[N:10]([C:11](=[O:12])[c:13]2[cH:14][cH:15][cH:16][cH:17][cH:18]2)[c:19]2[cH:20][cH:21][c:22]([CH3:23])[cH:24][cH:25]2)[cH:26][cH:27]1
  变化原子映射编号: ['1', '2', '5', '6', '7', '8', '9', '10', '11', '16', '19', '22', '23', '26']
    原子 1: C (度数=1, H数=3)
    原子 2: C (度数=3, H数=0)
    原子 5: C (度数=3, H数=0)
    原子 6: N (度数=2, H数=1)
    原子 7: C (度数=2, H数=1)
    原子 8: C (度数=2, H数=1)
    原子 9: C (度数=2, H数=1)
    原子 10: N (度数=2, H数=0)
    原子 11: C (度数=3, H数=0)
    原子 16: C (度数=3, H数=0)


---

## Step 2: 编辑位点分类

### 核心思想

识别出变化原子后，LocalRetro 需要将这些变化细分为不同类型的**编辑操作**：

| 编辑类型 | 代码 | 含义 | 示例 |
|----------|------|------|------|
| **A** (Atom/Add LG) | `'A'` | 原子处添加离去基团 | NH₂ 加上 Boc 保护基 |
| **B** (Break bond) | `'B'` | 断裂化学键 | 酰胺键断裂 |
| **C** (Change bond) | `'C'` | 改变键类型 | 单键→双键 |
| **R** (Remote) | `'R'` | 远程效应原子 | 与编辑位点不直接相连的变化原子 |

在逆合成方向（retro）：
- **原子编辑 (Atom edit)**: 类型 A 和 R → 分配 **atom template**
- **键编辑 (Bond edit)**: 类型 B 和 C → 分配 **bond template**

### LocalRetro 源码对应
- 文件: `LocalTemplate/template_extract_utils.py` → `label_retro_edit_site()`

In [5]:
# ====== Step 2: 编辑位点分类 ======
# 对应 LocalRetro 源码: template_extract_utils.py → label_retro_edit_site()

def check_bond_break(pbond, rbond):
    """检查是否有键断裂：产物中有键，反应物中无键"""
    return pbond is not None and rbond is None

def check_bond_change(pbond, rbond):
    """检查键类型是否改变"""
    if pbond is None or rbond is None:
        return False
    return bond_to_smarts(pbond) != bond_to_smarts(rbond)

def check_atom_change(patom, ratom):
    """检查原子的邻居连接是否改变"""
    return atom_neighbors(patom) != atom_neighbors(ratom)

def label_retro_edit_site(products, reactants, edit_nums):
    """
    将变化原子分类为不同的编辑类型。
    
    对应 LocalRetro 源码: template_extract_utils.py → label_retro_edit_site()
    
    返回:
        grow_atoms: A 类型（原子处添加离去基团）
        broken_bonds: B 类型（断裂键）
        changed_bonds: C 类型（键类型改变）
        remote_atoms: R 类型（远程效应原子）
    """
    edit_nums = [int(n) for n in edit_nums]
    pmol = Chem.MolFromSmiles(products)
    rmol = Chem.MolFromSmiles(reactants)
    patom_map = {a.GetAtomMapNum(): a.GetIdx() for a in pmol.GetAtoms()}
    ratom_map = {a.GetAtomMapNum(): a.GetIdx() for a in rmol.GetAtoms()}
    
    used_atoms = set()
    grow_atoms = []      # A: 添加离去基团
    broken_bonds = []    # B: 断键
    changed_bonds = []   # C: 改变键类型
    
    # 1. 检测断键 (B)
    for a in edit_nums:
        for b in edit_nums:
            if a >= b:
                continue
            if a not in patom_map or b not in patom_map:
                continue
            if a not in ratom_map or b not in ratom_map:
                continue
            pbond = pmol.GetBondBetweenAtoms(patom_map[a], patom_map[b])
            rbond = rmol.GetBondBetweenAtoms(ratom_map[a], ratom_map[b])
            if check_bond_break(pbond, rbond):
                broken_bonds.append((a, b))
                used_atoms.update([a, b])
    
    # 2. 检测离去基团添加 (A)
    for a in edit_nums:
        if a in used_atoms:
            continue
        if a not in patom_map or a not in ratom_map:
            continue
        patom = pmol.GetAtomWithIdx(patom_map[a])
        ratom = rmol.GetAtomWithIdx(ratom_map[a])
        if check_atom_change(patom, ratom):
            used_atoms.add(a)
            grow_atoms.append(a)
    
    # 3. 检测键类型改变 (C)
    for a in edit_nums:
        for b in edit_nums:
            if a >= b:
                continue
            if a not in patom_map or b not in patom_map:
                continue
            if a not in ratom_map or b not in ratom_map:
                continue
            pbond = pmol.GetBondBetweenAtoms(patom_map[a], patom_map[b])
            rbond = rmol.GetBondBetweenAtoms(ratom_map[a], ratom_map[b])
            if check_bond_change(pbond, rbond):
                if a not in used_atoms and b not in used_atoms:
                    changed_bonds.append((a, b))
                    changed_bonds.append((b, a))
    
    # 4. 识别远程原子 (R)
    all_used = set(grow_atoms + [a for bond in broken_bonds + changed_bonds for a in bond])
    remote_atoms = [a for a in edit_nums if a not in all_used and a in ratom_map]
    
    return grow_atoms, broken_bonds, changed_bonds, remote_atoms

In [6]:
# ====== 演示编辑位点分类 ======

print('编辑位点分类演示')
print('=' * 70)

for i, row in demo_data.iterrows():
    rxn = row['reactants>reagents>production']
    parts = rxn.split('>')
    reactant_smiles = parts[0]
    product_smiles = parts[-1]
    
    prod_mol = Chem.MolFromSmiles(product_smiles)
    reac_mols = [Chem.MolFromSmiles(s) for s in reactant_smiles.split('.') if Chem.MolFromSmiles(s)]
    
    _, changed_tags = get_changed_atoms(reac_mols, prod_mol)
    
    # 合并反应物为一个分子来进行编辑位点标注
    combined_reactants = '.'.join([Chem.MolToSmiles(m) for m in reac_mols])
    
    grow, broken, changed, remote = label_retro_edit_site(
        product_smiles, combined_reactants, changed_tags)
    
    print(f'\n反应 {i}:')
    print(f'  变化原子: {changed_tags}')
    print(f'  A (离去基团添加): {grow}')
    print(f'  B (断键):         {broken}')
    print(f'  C (键类型改变):   {changed}')
    print(f'  R (远程效应):     {remote}')

编辑位点分类演示

反应 0:
  变化原子: ['9', '10', '11']
  A (离去基团添加): []
  B (断键):         [(10, 11)]
  C (键类型改变):   []
  R (远程效应):     [9]

反应 1:
  变化原子: ['2', '3']
  A (离去基团添加): []
  B (断键):         [(2, 3)]
  C (键类型改变):   []
  R (远程效应):     []

反应 2:
  变化原子: ['1', '2', '5', '6', '7', '8', '9', '10', '11', '16', '19', '22', '23', '26']
  A (离去基团添加): [16, 26]
  B (断键):         [(1, 2), (5, 6), (8, 9), (10, 11), (10, 19), (22, 23)]
  C (键类型改变):   []
  R (远程效应):     [7]

反应 3:
  变化原子: ['2', '3']
  A (离去基团添加): []
  B (断键):         [(2, 3)]
  C (键类型改变):   []
  R (远程效应):     []

反应 4:
  变化原子: ['1', '2']
  A (离去基团添加): []
  B (断键):         [(1, 2)]
  C (键类型改变):   []
  R (远程效应):     []

反应 5:
  变化原子: ['1', '6', '7']
  A (离去基团添加): [1]
  B (断键):         []
  C (键类型改变):   [(6, 7), (7, 6)]
  R (远程效应):     []

反应 6:
  变化原子: ['6', '7']
  A (离去基团添加): []
  B (断键):         [(6, 7)]
  C (键类型改变):   []
  R (远程效应):     []

反应 7:
  变化原子: ['5', '6', '10']
  A (离去基团添加): [10]
  B (断键):         [(5, 6)]
  C (键类型改变):   []
  

---

## Step 3: 局部模板提取

### 什么是局部模板？

与 GLN 的全局模板（描述整个反应的变化）不同，LocalRetro 的**局部模板 (Local Template)** 只关注单个编辑位点周围的变化：

```
全局模板 (GLN):
  [C:1](=[O:2])[O:3][c:4]>>[C:1](=[O:2])[OH:3].[c:4][OH]
  （描述整个酯键断裂反应）

局部模板 (LocalRetro):
  键模板: [C:1]-[O:2]>>[C:1].[O:2]  + H_change={1:0, 2:1} + ...
  （只描述一个键的断裂 + 局部原子属性变化）
```

### 局部模板的组成

每个局部模板包含三部分信息：
1. **反应 SMARTS**: 描述键/原子变化的 SMARTS 模式
2. **H 变化 (H_change)**: 每个映射原子的氢原子数量变化
3. **电荷变化 (Charge_change)**: 每个映射原子的电荷变化
4. **手性变化 (Chiral_change)**: 每个映射原子的手性变化

完整的模板编码: `{SMARTS}_{H_code}_{Charge_code}_{Chiral_code}`

### LocalRetro 源码对应
- 文件: `LocalTemplate/template_extractor.py` → `extract_from_reaction()`
- 文件: `preprocessing/Extract_from_train_data.py` → `extract_templates()`

In [7]:
# ====== Step 3: 局部模板提取 (简化版) ======
# 对应 LocalRetro 源码: LocalTemplate/template_extractor.py

def get_local_template(pmol, rmol, edit_atoms, edit_type):
    """
    为一个编辑位点提取局部模板 SMARTS。
    
    简化版本，用于教学演示。实际 LocalRetro 源码的处理更复杂，
    包括环的处理、立体化学等。
    
    参数:
        pmol: 产物分子
        rmol: 反应物分子（合并后）
        edit_atoms: 编辑涉及的原子映射编号列表
        edit_type: 'A', 'B', 'C', 或 'R'
    
    返回:
        template_smarts: 局部模板 SMARTS
    """
    patom_map = {a.GetAtomMapNum(): a.GetIdx() for a in pmol.GetAtoms()}
    ratom_map = {a.GetAtomMapNum(): a.GetIdx() for a in rmol.GetAtoms()}
    
    # 构建产物端 SMARTS（只包含编辑原子及其近邻）
    if isinstance(edit_atoms, tuple):
        edit_list = list(edit_atoms)
    elif isinstance(edit_atoms, int):
        edit_list = [edit_atoms]
    else:
        edit_list = list(edit_atoms)
    
    # 简化处理：直接使用原子符号和映射编号构建 SMARTS
    prod_parts = []
    react_parts = []
    
    for atom_map in edit_list:
        if atom_map in patom_map:
            patom = pmol.GetAtomWithIdx(patom_map[atom_map])
            prod_parts.append(f'[{patom.GetSymbol()}:{atom_map}]')
        if atom_map in ratom_map:
            ratom = rmol.GetAtomWithIdx(ratom_map[atom_map])
            react_parts.append(f'[{ratom.GetSymbol()}:{atom_map}]')
    
    if edit_type == 'B':
        # 断键: 产物中有键，反应物中分开
        if len(edit_list) == 2:
            a, b = edit_list
            pbond = pmol.GetBondBetweenAtoms(patom_map[a], patom_map[b])
            if pbond:
                bond_char = {1: '-', 2: '=', 3: '#', 12: ':'}.get(
                    int(pbond.GetBondTypeAsDouble()*2)/2, '-')
                if pbond.GetBondTypeAsDouble() == 1.0:
                    bond_char = '-'
                elif pbond.GetBondTypeAsDouble() == 2.0:
                    bond_char = '='
                elif pbond.GetBondTypeAsDouble() == 1.5:
                    bond_char = ':'
                prod_smarts = f'[{pmol.GetAtomWithIdx(patom_map[a]).GetSymbol()}:{a}]{bond_char}[{pmol.GetAtomWithIdx(patom_map[b]).GetSymbol()}:{b}]'
            else:
                prod_smarts = '.'.join(prod_parts)
            react_smarts = '.'.join(react_parts)
        else:
            prod_smarts = '.'.join(prod_parts)
            react_smarts = '.'.join(react_parts)
    elif edit_type == 'A':
        prod_smarts = '.'.join(prod_parts)
        react_smarts = '.'.join(react_parts)
    else:
        prod_smarts = '.'.join(prod_parts)
        react_smarts = '.'.join(react_parts)
    
    return f'{prod_smarts}>>{react_smarts}'

def get_h_charge_change(pmol, rmol, edit_atoms):
    """
    计算编辑原子的 H/电荷/手性变化。
    
    对应 LocalRetro 源码: template_extract_utils.py → label_CHS_change()
    """
    patom_map = {a.GetAtomMapNum(): a for a in pmol.GetAtoms()}
    ratom_map = {a.GetAtomMapNum(): a for a in rmol.GetAtoms()}
    
    H_change = {}
    C_change = {}
    
    for atom_map in edit_atoms:
        atom_map = int(atom_map)
        if atom_map in patom_map and atom_map in ratom_map:
            pa = patom_map[atom_map]
            ra = ratom_map[atom_map]
            H_change[atom_map] = (ra.GetNumExplicitHs() + ra.GetNumImplicitHs()) - \
                                  (pa.GetNumExplicitHs() + pa.GetNumImplicitHs())
            C_change[atom_map] = ra.GetFormalCharge() - pa.GetFormalCharge()
    
    return H_change, C_change

def get_full_template_code(template_smarts, H_change, C_change):
    """
    将模板 SMARTS + H/电荷变化编码为完整的模板字符串。
    
    对应 LocalRetro 源码: Extract_from_train_data.py → get_full_template()
    
    格式: {SMARTS}_{H_code}_{Charge_code}
    """
    # 按映射编号排序
    sorted_keys = sorted(H_change.keys())
    H_code = ''.join([str(H_change[k]) for k in sorted_keys])
    C_code = ''.join([str(C_change[k]) for k in sorted_keys])
    return f'{template_smarts}_{H_code}_{C_code}'

In [8]:
# ====== 演示模板提取 ======
# 使用 LocalRetro 原始的 rdChiral 改版提取器（如果可用）
# 否则使用上面的简化版本

print('局部模板提取演示')
print('=' * 70)

all_templates = []

for i, row in demo_data.iterrows():
    rxn = row['reactants>reagents>production']
    parts = rxn.split('>')
    reactant_smiles = parts[0]
    product_smiles = parts[-1]
    
    prod_mol = Chem.MolFromSmiles(product_smiles)
    reac_mols = [Chem.MolFromSmiles(s) for s in reactant_smiles.split('.') if Chem.MolFromSmiles(s)]
    
    _, changed_tags = get_changed_atoms(reac_mols, prod_mol)
    combined_reactants = '.'.join([Chem.MolToSmiles(m) for m in reac_mols])
    rmol = Chem.MolFromSmiles(combined_reactants)
    
    grow, broken, changed, remote = label_retro_edit_site(
        product_smiles, combined_reactants, changed_tags)
    
    # 计算 H/电荷变化
    H_change, C_change = get_h_charge_change(prod_mol, rmol, changed_tags)
    
    print(f'\n反应 {i}:')
    print(f'  变化原子: {changed_tags}')
    print(f'  H 变化: {H_change}')
    print(f'  电荷变化: {C_change}')
    
    # 提取各类型的模板
    if grow:
        for a in grow:
            tpl = get_local_template(prod_mol, rmol, a, 'A')
            print(f'  原子模板 (A): {tpl}')
            all_templates.append(('atom', tpl, H_change, C_change))
    if broken:
        for bond in broken:
            tpl = get_local_template(prod_mol, rmol, bond, 'B')
            print(f'  键模板   (B): {tpl}')
            all_templates.append(('bond', tpl, H_change, C_change))
    if changed:
        for bond in changed[:len(changed)//2]:  # 避免重复 (a,b) 和 (b,a)
            tpl = get_local_template(prod_mol, rmol, bond, 'C')
            print(f'  键模板   (C): {tpl}')
            all_templates.append(('bond', tpl, H_change, C_change))

局部模板提取演示

反应 0:
  变化原子: ['9', '10', '11']
  H 变化: {9: 1, 10: 0, 11: 1}
  电荷变化: {9: 0, 10: 0, 11: 0}
  键模板   (B): [C:10]-[N:11]>>[C:10].[N:11]

反应 1:
  变化原子: ['2', '3']
  H 变化: {2: 0, 3: 1}
  电荷变化: {2: 0, 3: 0}
  键模板   (B): [C:2]-[N:3]>>[C:2].[N:3]

反应 2:
  变化原子: ['1', '2', '5', '6', '7', '8', '9', '10', '11', '16', '19', '22', '23', '26']
  H 变化: {1: 0, 2: 0, 5: 0, 6: 1, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 16: -1, 19: 0, 22: 0, 23: 0, 26: -1}
  电荷变化: {1: 0, 2: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 16: 0, 19: 0, 22: 0, 23: 0, 26: 0}
  原子模板 (A): [C:16]>>[C:16]
  原子模板 (A): [C:26]>>[C:26]
  键模板   (B): [C:1]-[C:2]>>[C:1].[C:2]
  键模板   (B): [C:5]-[N:6]>>[C:5].[N:6]
  键模板   (B): [C:8]=[C:9]>>[C:8].[C:9]
  键模板   (B): [N:10]-[C:11]>>[N:10].[C:11]
  键模板   (B): [N:10]-[C:19]>>[N:10].[C:19]
  键模板   (B): [C:22]-[C:23]>>[C:22].[C:23]

反应 3:
  变化原子: ['2', '3']
  H 变化: {2: 0, 3: 0}
  电荷变化: {2: -1, 3: 0}
  键模板   (B): [O:2]-[C:3]>>[O:2].[C:3]

反应 4:
  变化原子: ['1', '2']
  H 变化: {1: -1, 2: 0}
  电荷变化: 

---

## Step 4: 模板统计与编号

### 构建模板词典

从训练集中提取所有模板后，LocalRetro 将它们分为两类并编号：

1. **原子模板 (atom_templates.csv)**: 类型 A 和 R 的编辑使用
2. **键模板 (bond_templates.csv)**: 类型 B 和 C 的编辑使用

每个模板分配一个从 1 开始的**类别编号 (Class)**。编号 0 保留给「无模板」（即该位置不需要编辑）。

### 额外信息表 (template_infos.csv)

除了模板本身，还需要记录每个模板的：
- **edit_site**: 模板涉及的原子映射关系
- **change_H / change_C / change_S**: H 数/电荷/手性变化
- **Frequency**: 在训练集中出现的频次

这些信息在解码阶段（从预测的模板恢复反应物）时非常关键。

### LocalRetro 源码对应
- 文件: `preprocessing/Extract_from_train_data.py` → `extract_templates()`, `export_template()`

In [9]:
# ====== Step 4: 模板统计与编号 ======
# 对应 LocalRetro 源码: preprocessing/Extract_from_train_data.py

# 统计模板
atom_template_counter = Counter()
bond_template_counter = Counter()

for tpl_type, tpl_smarts, h_chg, c_chg in all_templates:
    if tpl_type == 'atom':
        atom_template_counter[tpl_smarts] += 1
    else:
        bond_template_counter[tpl_smarts] += 1

print('原子模板统计:')
print('-' * 40)
atom_template_dict = {}
for idx, (tpl, count) in enumerate(atom_template_counter.most_common()):
    atom_template_dict[tpl] = idx + 1  # 从1开始编号
    print(f'  Class {idx+1}: {tpl} (频次: {count})')

print(f'\n键模板统计:')
print('-' * 40)
bond_template_dict = {}
for idx, (tpl, count) in enumerate(bond_template_counter.most_common()):
    bond_template_dict[tpl] = idx + 1
    print(f'  Class {idx+1}: {tpl} (频次: {count})')

print(f'\n共 {len(atom_template_dict)} 个原子模板, {len(bond_template_dict)} 个键模板')
print(f'\n实际 USPTO_50K 数据集: 124 个原子模板, 548 个键模板')

原子模板统计:
----------------------------------------
  Class 1: [N:1]>>[N:1] (频次: 13)
  Class 2: [C:1]>>[C:1] (频次: 13)
  Class 3: [O:3]>>[O:3] (频次: 12)
  Class 4: [O:15]>>[O:15] (频次: 11)
  Class 5: [O:1]>>[O:1] (频次: 9)
  Class 6: [O:21]>>[O:21] (频次: 9)
  Class 7: [O:27]>>[O:27] (频次: 8)
  Class 8: [O:7]>>[O:7] (频次: 8)
  Class 9: [O:19]>>[O:19] (频次: 8)
  Class 10: [O:5]>>[O:5] (频次: 8)
  Class 11: [O:18]>>[O:18] (频次: 7)
  Class 12: [O:24]>>[O:24] (频次: 7)
  Class 13: [N:8]>>[N:8] (频次: 7)
  Class 14: [O:26]>>[O:26] (频次: 6)
  Class 15: [N:17]>>[N:17] (频次: 6)
  Class 16: [O:9]>>[O:9] (频次: 6)
  Class 17: [O:8]>>[O:8] (频次: 6)
  Class 18: [N:20]>>[N:20] (频次: 6)
  Class 19: [C:16]>>[C:16] (频次: 5)
  Class 20: [O:10]>>[O:10] (频次: 5)
  Class 21: [C:5]>>[C:5] (频次: 5)
  Class 22: [O:11]>>[O:11] (频次: 5)
  Class 23: [O:14]>>[O:14] (频次: 5)
  Class 24: [C:6]>>[C:6] (频次: 5)
  Class 25: [C:9]>>[C:9] (频次: 5)
  Class 26: [O:37]>>[O:37] (频次: 5)
  Class 27: [C:13]>>[C:13] (频次: 5)
  Class 28: [O:25]>>[O:25] (频次: 5)


---

## Step 5: 产物标注（为每个原子/键分配标签）

### 核心思想

这是 LocalRetro 数据处理中**最关键的一步**：将模板分类问题转化为原子/键级别的分类任务。

对于产物分子中的每一个：
- **原子**: 分配一个 atom template class（0 = 无编辑）
- **键**: 分配一个 bond template class（0 = 无编辑）

这样模型的任务就变成了：给定一个产物分子图，为每个原子和每个键预测其模板类别。

```
产物分子: CC(=O)Nc1ccccc1 (乙酰苯胺)
                     
原子标签:  [0, 0, 0, 1, 0, 0, 0, 0, 0]  ← 第3个原子(N)被标注为 atom_template_1
键标签:    [0, 0, 2, 0, 0, 0, 0, ...]    ← 第2条键(C-N)被标注为 bond_template_2
```

> **注意**: 大多数原子/键的标签都是 0（不参与反应），这使得类别分布**高度不平衡**。

### 编辑位点的确定

产物中有哪些可编辑的位点？
- **原子编辑位点**: 所有原子 `[0, 1, 2, ..., n_atoms-1]`
- **键编辑位点**: 所有有向边 `[(u,v), (v,u), ...]`（注意每条键有两个方向！）

### LocalRetro 源码对应
- 文件: `preprocessing/Run_preprocessing.py` → `labeling_dataset()`, `get_edit_site_retro()`

In [10]:
# ====== Step 5: 产物标注 ======
# 对应 LocalRetro 源码: preprocessing/Run_preprocessing.py → labeling_dataset()

def get_edit_site_retro(smiles):
    """
    获取产物分子的所有可编辑位点。
    
    对应 LocalRetro 源码: Run_preprocessing.py → get_edit_site_retro()
    
    返回:
        A: 原子位点列表 [0, 1, 2, ..., n_atoms-1]
        B: 键位点列表 [(u,v), (v,u), ...] (有向边)
    """
    mol = Chem.MolFromSmiles(smiles)
    A = list(range(mol.GetNumAtoms()))
    B = []
    for bond in mol.GetBonds():
        u = bond.GetBeginAtom().GetIdx()
        v = bond.GetEndAtom().GetIdx()
        B += [(u, v), (v, u)]
    return A, B

def label_product(product_smiles, reactant_smiles, changed_tags, 
                  grow_atoms, broken_bonds, changed_bonds, remote_atoms,
                  atom_template_dict, bond_template_dict, template_smarts_map):
    """
    为产物的每个原子/键分配模板标签。
    
    对应 LocalRetro 源码: Run_preprocessing.py → labeling_dataset() 中的标注逻辑
    
    返回:
        labels: 标签列表 [('a', atom_idx, template_class), ('b', bond_idx, template_class), ...]
    """
    prod_mol = Chem.MolFromSmiles(product_smiles)
    patom_map = {a.GetAtomMapNum(): a.GetIdx() for a in prod_mol.GetAtoms()}
    
    atom_sites, bond_sites = get_edit_site_retro(product_smiles)
    labels = []
    
    # A 类型: 原子处添加离去基团
    for atom_map in grow_atoms:
        if atom_map in patom_map:
            atom_idx = patom_map[atom_map]
            tpl_smarts = template_smarts_map.get(('A', atom_map), None)
            if tpl_smarts and tpl_smarts in atom_template_dict:
                labels.append(('a', atom_sites.index(atom_idx), atom_template_dict[tpl_smarts]))
    
    # R 类型: 远程原子
    for atom_map in remote_atoms:
        if atom_map in patom_map:
            atom_idx = patom_map[atom_map]
            tpl_smarts = template_smarts_map.get(('R', atom_map), None)
            if tpl_smarts and tpl_smarts in atom_template_dict:
                labels.append(('a', atom_sites.index(atom_idx), atom_template_dict[tpl_smarts]))
    
    # B 类型: 断键
    for (a, b) in broken_bonds:
        if a in patom_map and b in patom_map:
            bond_idx_pair = (patom_map[a], patom_map[b])
            tpl_smarts = template_smarts_map.get(('B', (a, b)), None)
            if tpl_smarts and tpl_smarts in bond_template_dict:
                if bond_idx_pair in bond_sites:
                    labels.append(('b', bond_sites.index(bond_idx_pair), bond_template_dict[tpl_smarts]))
    
    # C 类型: 键类型改变
    for (a, b) in changed_bonds:
        if a in patom_map and b in patom_map:
            bond_idx_pair = (patom_map[a], patom_map[b])
            tpl_smarts = template_smarts_map.get(('C', (a, b)), None)
            if tpl_smarts and tpl_smarts in bond_template_dict:
                if bond_idx_pair in bond_sites:
                    labels.append(('b', bond_sites.index(bond_idx_pair), bond_template_dict[tpl_smarts]))
    
    return labels

In [11]:
# ====== 演示产物标注 ======

print('产物标注演示')
print('=' * 70)

labeled_data = []

for i, row in demo_data.iterrows():
    rxn = row['reactants>reagents>production']
    parts = rxn.split('>')
    reactant_smiles = parts[0]
    product_smiles = parts[-1]
    
    prod_mol = Chem.MolFromSmiles(product_smiles)
    reac_mols = [Chem.MolFromSmiles(s) for s in reactant_smiles.split('.') if Chem.MolFromSmiles(s)]
    combined_reactants = '.'.join([Chem.MolToSmiles(m) for m in reac_mols])
    rmol = Chem.MolFromSmiles(combined_reactants)
    
    _, changed_tags = get_changed_atoms(reac_mols, prod_mol)
    grow, broken, changed, remote = label_retro_edit_site(
        product_smiles, combined_reactants, changed_tags)
    
    # 获取编辑位点信息
    atom_sites, bond_sites = get_edit_site_retro(product_smiles)
    
    print(f'\n反应 {i}:')
    print(f'  产物: {product_smiles}')
    print(f'  原子编辑位点: {atom_sites}')
    print(f'  键编辑位点 (前8个): {bond_sites[:8]}...')
    print(f'  A (grow atoms): {grow}')
    print(f'  B (broken bonds): {broken}')
    
    # 展示标签分配
    n_atoms = prod_mol.GetNumAtoms()
    n_bonds = len(bond_sites)
    atom_labels = [0] * n_atoms
    bond_labels = [0] * n_bonds
    
    # 简化标注（展示哪些位置被标注）
    patom_map = {a.GetAtomMapNum(): a.GetIdx() for a in prod_mol.GetAtoms()}
    
    for atom_map in grow:
        if atom_map in patom_map:
            idx = patom_map[atom_map]
            atom_labels[idx] = 1  # 示意：标记为有编辑
    
    for atom_map in remote:
        if atom_map in patom_map:
            idx = patom_map[atom_map]
            atom_labels[idx] = 1
    
    for (a, b) in broken:
        if a in patom_map and b in patom_map:
            pair = (patom_map[a], patom_map[b])
            if pair in bond_sites:
                bond_labels[bond_sites.index(pair)] = 1
    
    for (a, b) in changed:
        if a in patom_map and b in patom_map:
            pair = (patom_map[a], patom_map[b])
            if pair in bond_sites:
                bond_labels[bond_sites.index(pair)] = 1
    
    print(f'  原子标签: {atom_labels}  (非零 = 有编辑)')
    print(f'  键标签:   {bond_labels}  (非零 = 有编辑)')
    print(f'  标注覆盖: {sum(1 for x in atom_labels if x > 0)}/{n_atoms} 原子, '
          f'{sum(1 for x in bond_labels if x > 0)}/{n_bonds} 键')

产物标注演示

反应 0:
  产物: [CH3:1][c:2]1[n:3][c:4]2[c:5]([s:6]1)[C:7](=[O:8])/[C:9](=[CH:10]\[N:11]([CH3:12])[CH3:13])[CH2:14][CH2:15]2
  原子编辑位点: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
  键编辑位点 (前8个): [(0, 1), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2), (3, 4), (4, 3)]...
  A (grow atoms): []
  B (broken bonds): [(10, 11)]
  原子标签: [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]  (非零 = 有编辑)
  键标签:   [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  (非零 = 有编辑)
  标注覆盖: 1/15 原子, 1/32 键

反应 1:
  产物: [O:1]=[C:2]([NH:3][CH:4]1[CH:5]=[CH:6][CH:7]([OH:8])[CH2:9][CH2:10]1)[O:11][CH2:12][c:13]1[cH:14][cH:15][cH:16][cH:17][cH:18]1
  原子编辑位点: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
  键编辑位点 (前8个): [(0, 1), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2), (3, 4), (4, 3)]...
  A (grow atoms): []
  B (broken bonds): [(2, 3)]
  原子标签: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  (非零 = 有编辑)
  键标签:   [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

---

## Step 6: DGL 分子图构建

### 特征化方案

LocalRetro 使用 DGLLife 内置的特征化工具，与 GLN 手工编码特征不同：

| 特征化器 | 说明 | 维度 |
|----------|------|------|
| **WeaveAtomFeaturizer** | 原子特征：原子类型、度数、氢数、化合价等 | 27 维 |
| **CanonicalBondFeaturizer** | 键特征：键类型、共轭、环、立体 | 12 维 |

### WeaveAtomFeaturizer 特征分解

| 特征 | 维度 | 说明 |
|------|------|------|
| 原子类型 one-hot | 64+1 | 64种元素 + 未知 |
| 手性 one-hot | 4+1 | 4种手性类型 + 未指定 |
| 度数 one-hot | 6+1 | 0-6 度 |
| 电荷 one-hot | 5+1 | -2到+2 |
| 氢数 one-hot | 5+1 | 0-4 |
| 芳香性 | 1 | 0/1 |

### CanonicalBondFeaturizer 特征分解

| 特征 | 维度 | 说明 |
|------|------|------|
| 键类型 one-hot | 4 | 单/双/三/芳香 |
| 共轭 | 1 | 0/1 |
| 在环中 | 1 | 0/1 |
| 立体 one-hot | 6 | 6种立体类型 |

### 图构建

LocalRetro 使用 `smiles_to_bigraph` 将 SMILES 转换为 DGL 图，并添加**自环 (self-loop)**。自环是 MPNN 消息传递中包含节点自身信息的关键。

### LocalRetro 源码对应
- 文件: `scripts/utils.py` → `init_featurizer()`
- 文件: `scripts/dataset.py` → `USPTODataset._pre_process()`

In [12]:
# ====== Step 6: DGL 分子图构建 ======
# 对应 LocalRetro 源码: scripts/utils.py → init_featurizer() + dataset.py

import dgl
from dgllife.utils import (
    WeaveAtomFeaturizer, 
    CanonicalBondFeaturizer, 
    smiles_to_bigraph
)
from functools import partial

# LocalRetro 使用的原子类型列表（64 种元素）
atom_types = ['C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na', 'Ca', 'Fe',
         'As', 'Al', 'I', 'B', 'V', 'K', 'Tl', 'Yb', 'Sb', 'Sn', 'Ag', 'Pd', 'Co', 'Se', 'Ti',
         'Zn', 'H', 'Li', 'Ge', 'Cu', 'Au', 'Ni', 'Cd', 'In', 'Mn', 'Zr', 'Cr', 'Pt', 'Hg', 'Pb',
         'W', 'Ru', 'Nb', 'Re', 'Te', 'Rh', 'Ta', 'Tc', 'Ba', 'Bi', 'Hf', 'Mo', 'U', 'Sm', 'Os', 'Ir',
         'Ce', 'Gd', 'Ga', 'Cs']

# 创建特征化器
node_featurizer = WeaveAtomFeaturizer(atom_types=atom_types)
edge_featurizer = CanonicalBondFeaturizer(self_loop=True)

print(f'节点特征维度: {node_featurizer.feat_size()}')
print(f'边特征维度: {edge_featurizer.feat_size()}')

节点特征维度: 80
边特征维度: 13


In [13]:
# ====== 构建示例分子图 ======

smiles_to_graph = partial(smiles_to_bigraph, add_self_loop=True)

# 用第一个反应的产物构建 DGL 图
demo_smiles = 'CC(=O)Nc1ccccc1'  # 乙酰苯胺（去掉原子映射后）
graph = smiles_to_graph(demo_smiles, 
                        node_featurizer=node_featurizer,
                        edge_featurizer=edge_featurizer,
                        canonical_atom_order=False)

print(f'分子: {demo_smiles}')
print(f'节点数: {graph.num_nodes()} (原子数)')
print(f'边数:   {graph.num_edges()} (包括自环)')
print(f'\n节点特征 shape: {graph.ndata["h"].shape}')
print(f'边特征 shape:   {graph.edata["e"].shape}')
print()

# 查看具体特征
mol = Chem.MolFromSmiles(demo_smiles)
print('各原子特征 (前10维):')
for i, atom in enumerate(mol.GetAtoms()):
    feat = graph.ndata['h'][i][:10].numpy()
    print(f'  原子 {i} ({atom.GetSymbol()}): {feat}')

分子: CC(=O)Nc1ccccc1
节点数: 10 (原子数)
边数:   30 (包括自环)

节点特征 shape: torch.Size([10, 80])
边特征 shape:   torch.Size([30, 13])

各原子特征 (前10维):
  原子 0 (C): [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  原子 1 (C): [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  原子 2 (O): [0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
  原子 3 (N): [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
  原子 4 (C): [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  原子 5 (C): [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  原子 6 (C): [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  原子 7 (C): [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  原子 8 (C): [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  原子 9 (C): [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [14]:
# ====== 批量处理 & DataLoader 构建 ======
# 对应 LocalRetro 源码: scripts/utils.py → collate_molgraphs()

import torch

def collate_molgraphs(data):
    """
    将多个样本打包为一个 batch。
    
    对应 LocalRetro 源码: utils.py → collate_molgraphs()
    
    DGL 的 batch 操作将多个图合并为一个大的断连图 (disconnected graph),
    自动处理节点/边的索引偏移。
    """
    smiles, graphs, labels, masks = map(list, zip(*data))
    
    # 构建原子/键标签
    atom_labels = []
    bond_labels = []
    for g, label, m in zip(graphs, labels, masks):
        n_atoms = g.number_of_nodes()
        n_bonds = g.number_of_edges()
        atom_label = [0] * n_atoms
        bond_label = [0] * (n_bonds - n_atoms)  # 减去自环
        
        if m == 1:  # mask=1 表示有效样本
            for l in label:
                label_type, label_idx, label_template = l
                if label_type == 'a':
                    atom_label[label_idx] = label_template
                else:
                    bond_label[label_idx] = label_template
        
        atom_labels += atom_label
        bond_labels += bond_label
    
    bg = dgl.batch(graphs)
    return smiles, bg, torch.LongTensor(atom_labels), torch.LongTensor(bond_labels)

# 演示 batch 操作
smiles_list = ['CC(=O)Nc1ccccc1', 'c1ccc(-c2ccccc2)cc1', 'CC(=O)Oc1ccccc1']
graphs = [smiles_to_graph(s, node_featurizer=node_featurizer, 
                          edge_featurizer=edge_featurizer,
                          canonical_atom_order=False) for s in smiles_list]

bg = dgl.batch(graphs)
print(f'Batch 图统计:')
print(f'  总节点数: {bg.num_nodes()} (= {" + ".join([str(g.num_nodes()) for g in graphs])})')
print(f'  总边数:   {bg.num_edges()} (= {" + ".join([str(g.num_edges()) for g in graphs])})')
print(f'  节点特征: {bg.ndata["h"].shape}')
print(f'  边特征:   {bg.edata["e"].shape}')
print(f'\n  各子图节点数: {[g.num_nodes() for g in dgl.unbatch(bg)]}')

Batch 图统计:
  总节点数: 32 (= 10 + 12 + 10)
  总边数:   98 (= 30 + 38 + 30)
  节点特征: torch.Size([32, 80])
  边特征:   torch.Size([98, 13])

  各子图节点数: [10, 12, 10]


---

## 完整流程总结

### 数据处理管线回顾

| Step | 任务 | 输入 | 输出 | LocalRetro 源文件 |
|------|------|------|------|-------------------|
| 1 | 变化原子识别 | 反应 SMILES (带映射) | changed_atom_tags | `template_extractor.py` |
| 2 | 编辑位点分类 | 变化原子 + 分子 | A/B/C/R 编辑类型 | `template_extract_utils.py` |
| 3 | 局部模板提取 | 编辑位点 + 反应物产物 | 模板 SMARTS + H/C 变化 | `template_extractor.py` |
| 4 | 模板统计编号 | 所有模板 | `atom_templates.csv` + `bond_templates.csv` | `Extract_from_train_data.py` |
| 5 | 产物标注 | 产物 + 模板词典 | `labeled_data.csv` (每个原子/键的标签) | `Run_preprocessing.py` |
| 6 | DGL 图构建 | 产物 SMILES | DGL graph + 节点/边特征 | `dataset.py` + `utils.py` |

### 与 GLN 数据处理的关键区别

| 方面 | GLN | LocalRetro |
|------|-----|------------|
| 模板粒度 | 全局模板（整个反应） | **局部模板**（原子/键级别） |
| 模板表示 | `[prod_center]>>[react_center]` | `SMARTS_Hcode_Ccode` |
| 标签形式 | (产物, 正确模板) + 负采样 | 每个原子/键 → template_class |
| 图框架 | torch-geometric (scatter) | **DGL** (smiles_to_bigraph) |
| 特征化 | 手工 bit-packing + C++ 展开 | **DGLLife 内置** Featurizer |
| 训练目标 | 层次化概率：P(c|p)·P(t|c,p)·P(r|t,p) | **多分类**: 原子分类 + 键分类 |

### 关键数据文件

| 文件 | 内容 | 大小 (USPTO_50K) |
|------|------|------------------|
| `atom_templates.csv` | 原子级局部模板 | 124 个 |
| `bond_templates.csv` | 键级局部模板 | 548 个 |
| `template_infos.csv` | 模板详细信息 | 658 个 |
| `labeled_data.csv` | 标注后的训练/验证/测试数据 | ~50K 条 |

**下一步** → 请打开 `3_模型展示.ipynb`，学习 LocalRetro 的模型架构与推理流程。